# Linear Regression on Toy Dataset
Suppose we want to predict a house's Price ($y$) based on its Size (sq ft) ($X_1$) and Bedrooms ($X_2$):

In [11]:
import pandas as pd

data = {'Size': [1000, 1500, 2000], 'Bedrooms': [2, 3, 4], 'Price': [300, 400, 500]}
df = pd.DataFrame(data)



In [12]:
df

,Size,Bedrooms,Price
0,1000,2,300
1,1500,3,400
2,2000,4,500


In [13]:
X = df[['Size', 'Bedrooms']]  # 3 rows, 2 columns
y = df['Price']  # 3 values

In [16]:
X

,Size,Bedrooms
0,1000,2
1,1500,3
2,2000,4


In [17]:
y

0    300
1    400
2    500
Name: Price, dtype: int64

In [21]:
class CustomLinearRegression:

  def __init__(self):
    self.coef_ = None
    self.intercept_ = None
  def fit(self, X, y):
        # Convert input to numpy arrays
       X_mat = np.asarray(X, dtype=float)
       y_vec = np.asarray(y, dtype=float)

       # Add a column of ones for the intercept term (bias)
       ones = np.ones((X_mat.shape[0], 1)) # here np.ones(3,1)
       X_b = np.hstack([ones, X_mat])
       # OLS Normal Equation: beta = (X_b^T * X_b)^(-1) * X_b^T * y
       beta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y_vec
       # Split intercept and feature weights
       self.intercept_ = beta[0]
       self.coef_ = beta[1:]
       return self
  def predict(self, X):
       X_mat = np.asarray(X, dtype=float)
       return X_mat @ self.coef_ + self.intercept_

  def score(self, X, y):
  # Calculate Coefficient of Determination (R^2)
      y_true = np.asarray(y, dtype=float)
      y_pred = self.predict(X)

      ss_res = np.sum((y_true - y_pred) ** 2)
      ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

      return 1 - (ss_res / ss_tot)


In [22]:
model = CustomLinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)
print("R^2 Score:", model.score(X, y))

Intercept: -5.053051291908655e+18
Coefficients: [ 3.07863256e+15 -5.85467952e+18]
R^2 Score: -5.045759750791342e+34


# On automobile price Data

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
# Import
import requests, zipfile
import io

# Fetch automobile price data
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content

# Load the fetched data into a DataFrame object
auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)

# Set labels for the data columns
auto.columns =['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration', 'num-of-doors',
                            'body-style', 'drive-wheels', 'engine-location', 'wheel-base', 'length', 'width', 'height',
                            'curb-weight', 'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system', 'bore',
                            'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price']

In [5]:
auto = auto[['price', 'horsepower', 'width', 'height']]
auto.head()

,price,horsepower,width,height
0,13495,111,64.1,48.8
1,16500,111,64.1,48.8
2,16500,154,65.5,52.4
3,13950,102,66.2,54.3
4,17450,115,66.4,54.3


In [6]:
# Replace `?` with NaN and drop rows with NaN
auto = auto.replace('?', np.nan).dropna()
print('Shape of the car data:{}'.format(auto.shape))

Shape of the car data:(199, 4)


In [7]:
print('Check data types (before type conversion)\n{}\n'.format(auto.dtypes))

Check data types (before type conversion)
price          object
horsepower     object
width         float64
height        float64
dtype: object



In [8]:
auto = auto.assign(price=pd.to_numeric(auto.price))
auto = auto.assign(horsepower=pd.to_numeric(auto.horsepower))
print('Data type check (after type conversion\n{}'.format(auto.dtypes))

Data type check (after type conversion
price           int64
horsepower      int64
width         float64
height        float64
dtype: object


In [9]:
class CustomLinearRegression:

  def __init__(self):
    self.coef_ = None
    self.intercept_ = None

  def fit(self, X, y):
    # Convert input to numpy arrays
    X_mat = np.asarray(X, dtype=float)
    y_vec = np.asarray(y, dtype=float)

    # Add a column of ones for the intercept term (bias)
    ones = np.ones((X_mat.shape[0], 1))
    X_b = np.hstack([ones, X_mat])

    # OLS Normal Equation: beta = (X_b^T * X_b)^(-1) * X_b^T * y
    beta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y_vec

    # Split intercept and feature weights
    self.intercept_ = beta[0]
    self.coef_ = beta[1:]
    return self

  def predict(self, X):
    X_mat = np.asarray(X, dtype=float)
    return X_mat @ self.coef_ + self.intercept_

  def score(self, X, y):
    # Calculate Coefficient of Determination (R^2)
    y_true = np.asarray(y, dtype=float)
    y_pred = self.predict(X)

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    return 1 - (ss_res / ss_tot)

In [10]:
# Set `price` as target variable and the rest as explanatory variables
X = auto.drop('price', axis=1)
y = auto['price']

# Split into training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=0
)

# Initialize and train custom model
model = CustomLinearRegression()
model.fit(X_train, y_train)

# Display the coefficient of determination (R^2)
print(
    'Coefficient of determination (train): {:.3f}'.format(
        model.score(X_train, y_train)
    )
)
print(
    'Coefficient of determination (test): {:.3f}'.format(
        model.score(X_test, y_test)
    )
)

# Display regression coefficients and intercept
print(
    '\nRegression coefficients\n{}'.format(
        pd.Series(model.coef_, index=X.columns)
    )
)
print('Intercept: {:.3f}'.format(model.intercept_))


Coefficient of determination (train): 0.733
Coefficient of determination (test): 0.737

Regression coefficients
horsepower      81.651078
width         1829.174506
height         229.510077
dtype: float64
Intercept: -128409.046
